# Mizoram Jhum Detection Equation — v3 (1990–2025)
**Inputs (Google Drive, folder `Jhum_equation`)**
- `MIZ_Jhum_Samples_Landsat_1990_2025_v3.csv` – jhum + random background pixels (GEE Part A)
- `MIZ_Jhum_D3_ConfuserSamples_WorldCover_2020_2025_v3.csv` – built-up / bare / cropland pixels (GEE Part D) *(optional but recommended)*

**What this notebook does**
1. Loads and checks the data.
2. Compares candidate logistic equations with **leave-one-year-out** validation (does one equation work on an unseen year?).
3. Measures how often each equation wrongly flags **towns and bare land**.
4. Finds the best simple **two-threshold rule** (`NBR_post ≤ A` and `dNBR ≥ B`) with the same tests.
5. Fits the final equation (coefficients, 95% CI) and prints the `var EQ = {...}` block for GEE Part B.

> WorldCover cropland is **not** used for training or judging: in Mizoram it contains real jhum. Built-up (50) and bare (60) are used as confusers.

In [ ]:
# 1. Setup
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR   = '/content/drive/MyDrive/Jhum_equation'
SAMPLE_CSV = 'MIZ_Jhum_Samples_Landsat_1990_2025_v3.csv'
CONF_CSV   = 'MIZ_Jhum_D3_ConfuserSamples_WorldCover_2020_2025_v3.csv'
OUT_DIR    = os.path.join(DATA_DIR, 'MIZ_Jhum_Model_Outputs_v3')
os.makedirs(OUT_DIR, exist_ok=True)

MAX_PX_PER_POLY = 25     # stops large polygons dominating
CONF_CLASSES    = {50: 'built_up', 60: 'bare'}   # confusers used (cropland excluded)
MAX_CONF_FA     = 5.0    # target: at most 5 % of confuser pixels called jhum
RANDOM_SEED     = 42
print('Outputs ->', OUT_DIR)

In [ ]:
# 2. Libraries
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, f1_score
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 170)

## 3. Load samples and confusers

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, SAMPLE_CSV))
df = df.dropna(subset=['NBR_pre', 'NBR_post', 'dNBR'])
df = df[(df['n_pre'] > 0) & (df['n_post'] > 0)]
print('Samples:', len(df))
print(pd.crosstab(df['year'], df['sample_type'], margins=True))

conf_path = os.path.join(DATA_DIR, CONF_CSV)
if os.path.exists(conf_path):
    conf = pd.read_csv(conf_path).dropna(subset=['NBR_pre', 'NBR_post', 'dNBR'])
    conf = conf[conf['lc'].isin(CONF_CLASSES)].copy()
    conf['conf_class'] = conf['lc'].map(CONF_CLASSES)
    conf['jhum'] = 0
    print('\nConfuser pixels:')
    print(pd.crosstab(conf['year'], conf['conf_class']))
else:
    conf = None
    print('\nNo confuser file found – confuser tests will be skipped.')

## 4. Thin jhum pixels per polygon

In [ ]:
pos = (df[df['jhum'] == 1].groupby('poly_id', group_keys=False)
       .apply(lambda g: g.sample(min(len(g), MAX_PX_PER_POLY), random_state=RANDOM_SEED)))
bg = df[df['jhum'] == 0]
data = pd.concat([pos, bg]).reset_index(drop=True)
YEARS = sorted(data['year'].unique())
print(pd.crosstab(data['year'], data['jhum'], margins=True))

## 5. Signal check by year
Jhum should be low in NBR_post and high in dNBR in **every** year; background and confusers should not.

In [ ]:
tab = data.assign(group=np.where(data.jhum == 1, 'jhum', 'background'))
parts = [tab[['year', 'group', 'NBR_pre', 'NBR_post', 'dNBR']]]
if conf is not None:
    parts.append(conf.assign(group=conf['conf_class'])[['year', 'group', 'NBR_pre', 'NBR_post', 'dNBR']])
sig = pd.concat(parts).groupby(['year', 'group']).median().round(3)
sig.to_csv(os.path.join(OUT_DIR, 'MIZ_Jhum_v3_SignalByYear.csv'))
sig

## 6. Helper functions

In [ ]:
CANDIDATES = {
    'M1_NBRpost':          ['NBR_post'],
    'M2_dNBR':             ['dNBR'],
    'M3_NBRpost_dNBR':     ['NBR_post', 'dNBR'],
    'M4_NBRpre_NBRpost':   ['NBR_pre', 'NBR_post'],
    'M5_NBRpost_dNBR_slope': ['NBR_post', 'dNBR', 'slope'],
}

def fit_logit(d, cols):
    X = sm.add_constant(d[cols], has_constant='add')
    return sm.Logit(d['jhum'], X).fit(disp=0, maxiter=300)

def predict(m, d, cols):
    return np.asarray(m.predict(sm.add_constant(d[cols], has_constant='add')))

def youden_cut(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])

def metrics(y, pred, p=None):
    y = np.asarray(y); pred = np.asarray(pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    out = {'OA': (tp + tn) / len(y),
           'PA_jhum': tp / (tp + fn) if (tp + fn) else np.nan,
           'UA_jhum': tp / (tp + fp) if (tp + fp) else np.nan,
           'F1': f1_score(y, pred, zero_division=0)}
    if p is not None and len(np.unique(y)) > 1:
        out['AUC'] = roc_auc_score(y, p)
    return out

def conf_fa(pred_fn, years=None):
    # % of confuser pixels called jhum, per confuser class
    if conf is None: return {}
    c = conf if years is None else conf[conf.year.isin(years)]
    return {f'FA_{k}_%': 100 * np.mean(pred_fn(g)) for k, g in c.groupby('conf_class')}

has_conf = conf is not None

## 7. Leave-one-year-out: logistic equations
Train on six years (+ confusers from those years, if available), test on the unseen year.
Training **with** confusers teaches the equation to reject towns and bare land.

In [ ]:
rows = []
for name, cols in CANDIDATES.items():
    for yr in YEARS:
        tr = data[data.year != yr]
        if has_conf:
            tr = pd.concat([tr, conf[conf.year != yr][tr.columns.intersection(conf.columns)]])
        tr = tr.dropna(subset=cols)
        te = data[data.year == yr].dropna(subset=cols)
        try:
            m = fit_logit(tr, cols)
        except Exception as e:
            print(name, yr, 'failed:', e); continue
        cut = youden_cut(tr['jhum'], predict(m, tr, cols))
        p = predict(m, te, cols)
        row = {'model': name, 'test_year': yr, 'cutoff': cut, **metrics(te['jhum'], p >= cut, p)}
        if has_conf and 'slope' not in cols:
            row.update(conf_fa(lambda g: predict(m, g, cols) >= cut, years=[yr]))
        row.update({f'b_{k}': v for k, v in m.params.items()})
        rows.append(row)

loyo = pd.DataFrame(rows)
loyo.to_csv(os.path.join(OUT_DIR, 'MIZ_Jhum_v3_LOYO_Logistic.csv'), index=False)

agg_cols = [c for c in ['AUC', 'F1', 'PA_jhum', 'UA_jhum'] if c in loyo]
comp = loyo.groupby('model')[agg_cols].agg(['mean', 'min']).round(3)
comp.columns = ['_'.join(c) for c in comp.columns]
fa_cols = [c for c in loyo.columns if c.startswith('FA_')]
if fa_cols:
    comp = comp.join(loyo.groupby('model')[fa_cols].mean().round(1).add_suffix('_heldout'))
comp = comp.sort_values('F1_mean', ascending=False)
comp.to_csv(os.path.join(OUT_DIR, 'MIZ_Jhum_v3_ModelComparison.csv'))
comp

### Coefficient stability (flat lines = the same equation works in every year)

In [ ]:
BEST = 'M3_NBRpost_dNBR'      # recommended: state + change; change if the table above says otherwise
BEST_COLS = CANDIDATES[BEST]
sub = loyo[loyo.model == BEST].sort_values('test_year')
bcols = [c for c in sub.columns if c.startswith('b_') and sub[c].notna().all()]
fig, axes = plt.subplots(1, len(bcols), figsize=(4.5 * len(bcols), 3.5))
for ax, c in zip(np.atleast_1d(axes), bcols):
    ax.plot(sub['test_year'], sub[c], 'o-', color='#2e86ab')
    ax.axhline(sub[c].mean(), ls='--', color='grey')
    ax.set_title(c.replace('b_', 'β ')); ax.set_xlabel('Held-out year')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'MIZ_Jhum_v3_Fig_CoefStability_{BEST}.png'), dpi=300)
plt.show()
print(sub[['test_year'] + agg_cols + fa_cols + bcols].round(3).to_string(index=False))

## 8. Simple two-threshold rule: `NBR_post ≤ A` and `dNBR ≥ B`
Easy to explain and to apply to any Landsat year. Chosen by: highest mean F1 across years,
subject to confuser false alarms ≤ `MAX_CONF_FA` %. Then tested leave-one-year-out.

In [ ]:
A_GRID = np.round(np.arange(-0.05, 0.25, 0.01), 2)
B_GRID = np.round(np.arange(0.00, 0.55, 0.05), 2)

def rule(d, a, b):
    return ((d['NBR_post'] <= a) & (d['dNBR'] >= b)).values

def score_rule(d, a, b, conf_d=None):
    f1s = [metrics(g['jhum'], rule(g, a, b))['F1'] for _, g in d.groupby('year')]
    out = {'A': a, 'B': b, 'F1_mean': np.mean(f1s), 'F1_min': np.min(f1s)}
    if conf_d is not None and len(conf_d):
        out['conf_FA_%'] = 100 * np.mean(rule(conf_d, a, b))
    return out

def choose_rule(d, conf_d):
    g = pd.DataFrame([score_rule(d, a, b, conf_d) for a in A_GRID for b in B_GRID])
    if 'conf_FA_%' in g:
        ok = g[g['conf_FA_%'] <= MAX_CONF_FA]
        g = ok if len(ok) else g
    return g.sort_values(['F1_mean', 'F1_min'], ascending=False), g

grid_all, _ = choose_rule(data, conf)
grid_all.to_csv(os.path.join(OUT_DIR, 'MIZ_Jhum_v3_RuleGrid_AllYears.csv'), index=False)
print('Top rules (all years):'); print(grid_all.head(10).round(3).to_string(index=False))

rrows = []
for yr in YEARS:
    ctr = conf[conf.year != yr] if has_conf else None
    best_tr = choose_rule(data[data.year != yr], ctr)[0].iloc[0]
    te = data[data.year == yr]
    r = {'test_year': yr, 'A_NBRpost': best_tr['A'], 'B_dNBR': best_tr['B'],
         **metrics(te['jhum'], rule(te, best_tr['A'], best_tr['B']))}
    if has_conf:
        r.update(conf_fa(lambda g: rule(g, best_tr['A'], best_tr['B']), years=[yr]))
    rrows.append(r)
rule_loyo = pd.DataFrame(rrows).round(3)
rule_loyo.to_csv(os.path.join(OUT_DIR, 'MIZ_Jhum_v3_LOYO_TwoThresholdRule.csv'), index=False)
print('\nLeave-one-year-out, two-threshold rule:'); print(rule_loyo.to_string(index=False))

## 9. Final logistic equation (all years + confusers)

In [ ]:
train_all = data.copy()
if has_conf:
    train_all = pd.concat([train_all, conf[train_all.columns.intersection(conf.columns)]])
train_all = train_all.dropna(subset=BEST_COLS)

final = fit_logit(train_all, BEST_COLS)
print(final.summary())
CUT = youden_cut(train_all['jhum'], predict(final, train_all, BEST_COLS))

ci = final.conf_int(); ci.columns = ['CI95_low', 'CI95_high']
coef_tab = pd.concat([final.params.rename('coef'), final.bse.rename('SE'),
                      final.pvalues.rename('p_value'), ci], axis=1)
coef_tab.to_csv(os.path.join(OUT_DIR, f'MIZ_Jhum_v3_FinalEquation_{BEST}.csv'))
print(coef_tab.round(4)); print(f'\nCutoff (Youden): {CUT:.4f}')

p_all = predict(final, data, BEST_COLS)
per_year = pd.DataFrame([{'year': y, **metrics(data.loc[data.year == y, 'jhum'],
                                               p_all[data.year.values == y] >= CUT)} for y in YEARS]).round(3)
print('\nIn-sample per year (independent accuracy = section 7):'); print(per_year.to_string(index=False))
if has_conf:
    print('\nConfuser false alarms (%):', {k: round(v, 1) for k, v in
          conf_fa(lambda g: predict(final, g, BEST_COLS) >= CUT).items()})

## 10. Equation text + block to paste into GEE Part B

In [ ]:
b = final.params
terms = ''.join(f" {'+' if b[c] >= 0 else '−'} {abs(b[c]):.4f}·{c}" for c in BEST_COLS)
eq_text = f"z = {b['const']:.4f}{terms};  P(jhum) = 1/(1+exp(−z));  jhum if P ≥ {CUT:.3f}"

best_rule = grid_all.iloc[0]
rule_text = f"Rule: jhum if NBR_post ≤ {best_rule['A']:.2f} and dNBR ≥ {best_rule['B']:.2f}"

eq_js = ("var EQ = {\n"
         f"  model: '{BEST}_v3',\n"
         f"  b0: {b['const']:.6f},\n"
         "  coef: {" + ', '.join(f'{c}: {b[c]:.6f}' for c in BEST_COLS) + "},\n"
         f"  cut: {CUT:.4f},\n"
         "  min_dNBR: 0\n"
         "};")

print(eq_text); print(rule_text); print(); print(eq_js)
with open(os.path.join(OUT_DIR, f'MIZ_Jhum_v3_Equation_for_GEE_{BEST}.txt'), 'w', encoding='utf-8') as f:
    f.write(eq_text + '\n' + rule_text + '\n\n' + eq_js + '\n')
print('\nSaved to', OUT_DIR)